In [ ]:
import requests
from bs4 import BeautifulSoup
import pprint

url = "https://webscraper.io/test-sites/e-commerce/allinone"

print(f"Sending HTTP GET request to: {url}...")
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    print("\n[SUCCESS] Webpage downloaded and compiled into BeautifulSoup tree!")
else:
    print(f"\n[ERROR] Failed to fetch site. Server status code: {response.status_code}")
    
print(f"Soupskie: {soup.prettify()}")    

In [3]:
product_cards = soup.find_all('div', class_='thumbnail')

catalog = []

print(f"Processing data pipeline for {len(product_cards)} identified items...\n")

for card in product_cards:
    try:
        title = card.find('a', class_='title')['title']
        description = card.find('p', class_='description').text
        raw_price = card.find('h4', class_='price').text
        raw_reviews = card.find('p', class_='review-count').text
        
        # --- DATA CLEANING & TYPE TRANSFORMATION ---
        # Strip the currency symbol and parse the clean numerical string to a float
        clean_price = float(raw_price.replace('$', ''))
        
        # Convert string context ("12 reviews") into a pure integer data asset
        clean_reviews = int(raw_reviews.split()[0])
        
        # Build the final structured schema object
        product_record = {
            "name": title.strip(),
            "price_usd": clean_price,
            "description": description.strip(),
            "reviews_count": clean_reviews
        }
        
        # Load the record array
        catalog.append(product_record)
        
    except Exception as e:
        # Prevents an irregular or missing layout feature from breaking the entire operation
        print(f"[SKIP] Encountered unparseable item card structure. Error context: {e}")

# 3. Display the final pipeline dataset state
print(f"Pipeline Execution Complete! Ingested {len(catalog)} clean records.\n")
print("--- PREVIEWING TOP 2 SCHEMA RECORDS ---")
pprint.pprint(catalog[:2])

Processing data pipeline for 3 identified items...

Pipeline Execution Complete! Ingested 3 clean records.

--- PREVIEWING TOP 2 SCHEMA RECORDS ---
[{'description': 'Acer Predator Helios 300 (PH317-51), 17.3" FHD IPS, Core '
                 'i7-7700HQ. 8GB, 128GB SSD +1TB, GeForce GTX 1050Ti 4GB, '
                 'Linux + Windows 10 Home',
  'name': 'Acer Predator Helios 300 (PH317-51)',
  'price_usd': 1187.98,
  'reviews_count': 14},
 {'description': 'Lenovo Legion Y520, 15.6" FHD, Core i7-7700HQ, 8GB, 128 GB '
                 'SSD + 1TB HDD, GTX 1050 4GB, Windows 10 Home',
  'name': 'Lenovo Legion Y520',
  'price_usd': 1133.91,
  'reviews_count': 13}]
